# HWP → PDF/DOCX → Docling 파싱 노트북

이 노트북은 GitHub Codespaces/Ubuntu 환경을 기준으로 합니다.

권장 흐름:

1. `.hwp`를 LibreOffice + H2Orestart로 PDF로 변환
2. 변환된 PDF를 Docling으로 파싱
3. Markdown / TXT / Docling JSON / 조문 CSV / 표 CSV로 저장

> `.hwpx`도 같은 방식으로 시도할 수 있습니다. 변환이 깨지면 원본 HWP 버전, 암호화 여부, 폰트, 표 복잡도 문제일 수 있습니다.

In [1]:
# 1) 시스템/파이썬 패키지 설치
# Codespaces는 Ubuntu 기반이라 sudo apt-get 사용 가능성이 높습니다.

!sudo apt-get update -y
!sudo apt-get install -y libreoffice fonts-nanum fonts-noto-cjk fonts-unfonts-core poppler-utils

# HWP/HWPX LibreOffice import 확장: apt로 설치 가능한 환경이면 가장 간단합니다.
# 패키지가 없으면 아래 셀에서 GitHub release의 .oxt 확장을 받아 설치합니다.
!sudo apt-get install -y libreoffice-h2orestart || true

!python -m pip install -U pip
!python -m pip install -U docling pandas openpyxl tqdm

Get:1 https://packages.microsoft.com/repos/microsoft-ubuntu-noble-prod noble InRelease [3600 B]
Get:2 http://archive.ubuntu.com/ubuntu noble InRelease [256 kB]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 https://packages.microsoft.com/repos/microsoft-ubuntu-noble-prod noble/main all Packages [643 B]
Get:5 https://packages.microsoft.com/repos/microsoft-ubuntu-noble-prod noble/main amd64 Packages [177 kB]
Get:6 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [1308 kB]
Get:7 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [964 kB]
Get:8 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1486 kB]
Get:9 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [43.8 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]       
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble/m

In [2]:
# 2) H2Orestart 확장 fallback 설치
# apt 패키지가 없거나 변환이 실패할 경우를 대비해 최신 release의 .oxt를 설치합니다.

import json
import subprocess
import urllib.request
from pathlib import Path

def run(cmd, check=False):
    print("+", " ".join(map(str, cmd)))
    p = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    return p

oxt_path = Path("H2Orestart-latest.oxt")

try:
    with urllib.request.urlopen("https://api.github.com/repos/ebandal/H2Orestart/releases/latest", timeout=30) as resp:
        release = json.load(resp)

    assets = [
        a for a in release.get("assets", [])
        if a.get("name", "").lower().endswith(".oxt")
    ]

    if assets:
        url = assets[0]["browser_download_url"]
        print("Downloading:", assets[0]["name"], url)
        urllib.request.urlretrieve(url, oxt_path)
        run(["sudo", "unopkg", "add", "--shared", str(oxt_path)], check=False)
    else:
        print("No .oxt asset found in latest release. apt package may already be enough.")
except Exception as e:
    print("Fallback extension install skipped:", repr(e))

# LibreOffice 백그라운드 프로세스가 떠 있으면 변환 충돌이 날 수 있어 정리합니다.
run(["pkill", "-f", "soffice"], check=False)

Downloading: H2Orestart.oxt https://github.com/ebandal/H2Orestart/releases/download/v0.7.12/H2Orestart.oxt
+ sudo unopkg add --shared H2Orestart-latest.oxt



+ pkill -f soffice



CompletedProcess(args=['pkill', '-f', 'soffice'], returncode=1, stdout='')

In [4]:
# 3) 입력 파일 경로 설정
# Codespaces에서는 repo 안에 data/ 폴더를 만들고 HWP 파일을 넣는 방식을 추천합니다.
# 예: data/13.제규정 관리규정(규정 제12호)_2016.10.17.hwp

from pathlib import Path

INPUT = Path("data/13.제규정 관리규정(규정 제12호)_2016.10.17.hwp")

# ChatGPT 샌드박스에서 바로 테스트할 때만 아래처럼 사용할 수 있습니다.
# INPUT = Path("/mnt/data/13.제규정 관리규정(규정 제12호)_2016.10.17.hwp")

OUT_DIR = Path("parsed_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("input:", INPUT)
print("exists:", INPUT.exists())
if INPUT.exists():
    print("size:", INPUT.stat().st_size, "bytes")
else:
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {INPUT.resolve()}")

input: data/13.제규정 관리규정(규정 제12호)_2016.10.17.hwp
exists: True
size: 35840 bytes


In [5]:
# 4) HWP/HWPX → PDF 변환
# Docling은 HWP를 직접 입력으로 넣기보다 PDF/DOCX 등 지원 포맷으로 변환한 뒤 파싱하는 흐름이 안전합니다.

import subprocess
from pathlib import Path

DOCLING_SUPPORTED_DIRECT = {
    ".pdf", ".docx", ".xlsx", ".pptx",
    ".html", ".htm", ".md", ".csv",
    ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp",
}

def convert_hwp_to_pdf(input_path: Path, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)

    # --infilter=Hwp2002_File은 HWP 변환 품질을 올리는 데 도움이 되는 경우가 많습니다.
    cmd = [
        "libreoffice",
        "--headless",
        "--nologo",
        "--nofirststartwizard",
        "--infilter=Hwp2002_File",
        "--convert-to",
        "pdf:writer_pdf_Export",
        "--outdir",
        str(out_dir),
        str(input_path),
    ]

    print("+", " ".join(cmd))
    p = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)

    pdf_path = out_dir / f"{input_path.stem}.pdf"
    if not pdf_path.exists():
        raise RuntimeError(
            "PDF 변환 실패. H2Orestart 확장 설치 여부, 파일 암호화 여부, HWP 버전을 확인하세요.\n"
            f"expected: {pdf_path}"
        )
    return pdf_path

suffix = INPUT.suffix.lower()

if suffix in DOCLING_SUPPORTED_DIRECT:
    source_for_docling = INPUT
elif suffix in {".hwp", ".hwpx"}:
    source_for_docling = convert_hwp_to_pdf(INPUT, OUT_DIR)
else:
    raise ValueError(f"지원하지 않는 확장자입니다: {suffix}")

print("Docling input:", source_for_docling)

+ libreoffice --headless --nologo --nofirststartwizard --infilter=Hwp2002_File --convert-to pdf:writer_pdf_Export --outdir parsed_output data/13.제규정 관리규정(규정 제12호)_2016.10.17.hwp


convert /workspaces/search-lab/examples/data/13.제규정 관리규정(규정 제12호)_2016.10.17.hwp as a Writer document -> /workspaces/search-lab/examples/parsed_output/13.제규정 관리규정(규정 제12호)_2016.10.17.pdf using filter : writer_pdf_Export

Docling input: parsed_output/13.제규정 관리규정(규정 제12호)_2016.10.17.pdf


In [6]:
# 5) Docling으로 문서 파싱

from docling.document_converter import DocumentConverter

converter = DocumentConverter()
result = converter.convert(str(source_for_docling))
doc = result.document

preview = doc.export_to_markdown()[:3000]
print(preview)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[INFO] 2026-06-19 02:20:07,921 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-19 02:20:07,940 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-19 02:20:07,944 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-19 02:20:09,427 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-06-19 02:20:09,740 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/python/3.12.1/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-19 02:20:09,744 [RapidOCR] main.py:50: Using /usr/local/python/3.12.1/lib/python3.12/site-packages/rapi

## 세종특별자치시시설관리공단 제규정 관리규정

제정 (2016. 10. 17. 규정 제 12 호

## 제 1 장 총 칙

제 1 조 ( 목적 ) 이 규정은 세종특별자치시시설관리공단의 규정 ( 내규 포함 ) 의 체계와 그 제정 개정 폐지 시행 및 관리에 관한 사항을 정확히 정하는 등 제규정의 적정한 관리 ․ ․ ․ 운용에 필요한 사항을 규정함을 목적으로 한다 ․ .

제 2 조 ( 정의 ) 이 규정에서 사용하는 용어의 정의는 다음과 같다 .

1. ' 규정'이라 함은 세종특별자치시시설관리공단 ( 이하 '공단'이라 한다 ) 의 기본조직 , 경영활동의 질서 , 직원의 권리와 의무 , 제반업무수행 등에 관한 방침 및 기준으로서 체계적인 형식을 갖춘 공단규범의 근간이 되는 것을 말한다 .
2. '
3. 내규'라 함은 이 규정에서 위임한 사항과 공단업무 중 부분적이며 한정적인
4. 업무에 관한 사무처리방법 및 절차 등을 정한 것으로 규정보다 하위인 규범을 말한다 .
3. '
6. 규정안'이라 함은 이 규정에 의하여 규정을 제정 개정 및 폐지하기 위하여 작성한 ․ 안으로써 규정으로 확정되기 전까지의 것을 말한다 .
7. 소관부서'라 함은 규정에서 규율하고 있거나 규정으로 규율하고자 하는 사항을
4. ' 관장하는 부서를 말한다 .
5. ' 주관부서'라 함은 공단의 제규정의 관리를 주관하는 부서를 말한다 .
10. 제 3 조 ( 규정의 총괄 ) 주관부서는 법무를 담당하는 부서로 하며 , 당해 부서의 장은 공단의 제규정의 효율적인 관리를 위하여 필요한 경우 다음 각 호의 조치를 취할 수 있다 .
11. 1.
12. 소관부서에 대한 미제정된 규정의 제정 요구
2. 소관부서에 대한 현행규정의 폐지 정비요구 ․
3. 제 1 호 및 제 2 호의 요구에도 불구하고 소관부서에서 조치를 취하지 않는 규정의 제정 개정 및 폐지 ․
15. 4.
16. 기타 제규정의 관리운용에 관한 조정 통제 ․
17. 제 4 조 ( 규정의 분류 ) 규정은 그 성질과 내용에 따라 다음 각 호와 같이 분류한다

In [7]:
# 6) Markdown / TXT / JSON 저장

import json
from pathlib import Path

base = OUT_DIR / INPUT.stem

markdown = doc.export_to_markdown()
text = doc.export_to_text()
docling_json = doc.export_to_dict()

md_path = base.with_suffix(".md")
txt_path = base.with_suffix(".txt")
json_path = OUT_DIR / f"{INPUT.stem}.docling.json"

md_path.write_text(markdown, encoding="utf-8")
txt_path.write_text(text, encoding="utf-8")
json_path.write_text(json.dumps(docling_json, ensure_ascii=False, indent=2), encoding="utf-8")

print("saved:", md_path)
print("saved:", txt_path)
print("saved:", json_path)

saved: parsed_output/13.제규정 관리규정(규정 제12호)_2016.10.md
saved: parsed_output/13.제규정 관리규정(규정 제12호)_2016.10.txt
saved: parsed_output/13.제규정 관리규정(규정 제12호)_2016.10.17.docling.json


In [8]:
# 7) 규정/공문류 문서용: 조문 단위 CSV 만들기
# 예: 제1조(목적), 제2조(정의) ... 형태를 자동 분리합니다.

import re
import pandas as pd

text_for_articles = text

# 줄 시작의 제N조(제목)을 기준으로 분리
article_start = re.compile(r"(?=^제\s*\d+\s*조\s*\([^)\n]+\))", re.MULTILINE)
blocks = [b.strip() for b in article_start.split(text_for_articles) if b.strip()]

rows = []
for block in blocks:
    m = re.match(r"제\s*(\d+)\s*조\s*\(([^)\n]+)\)", block)
    if not m:
        continue
    article_no = int(m.group(1))
    title = m.group(2).strip()
    rows.append({
        "article_no": article_no,
        "title": title,
        "content": block,
        "char_len": len(block),
    })

articles_df = pd.DataFrame(rows)
articles_csv = OUT_DIR / f"{INPUT.stem}.articles.csv"
articles_xlsx = OUT_DIR / f"{INPUT.stem}.articles.xlsx"

articles_df.to_csv(articles_csv, index=False, encoding="utf-8-sig")
articles_df.to_excel(articles_xlsx, index=False)

print("articles:", len(articles_df))
display(articles_df.head(20))
print("saved:", articles_csv)
print("saved:", articles_xlsx)

articles: 9


,article_no,title,content,char_len
0,1,목적,제 1 조 ( 목적 ) 이 규정은 세종특별자치시시설관리공단의 규정 ( 내규 포함 )...,136
1,2,정의,제 2 조 ( 정의 ) 이 규정에서 사용하는 용어의 정의는 다음과 같다 .\n\n1...,895
2,5,효력,제 5 조 ( 효력 ) ① 법령 조례 정관 및 이사회 결의에 저촉되는 규정은 그 부...,144
3,6,개정형식 등,제 6 조 ( 개정형식 등 ) 규정을 개정하고자 하는 경우에는 다음 각 호의 사항을...,1341
4,10,절차,제 10 조 ( 절차 ) ① 규정의 제정 개정 및 폐지절차는 다음 각 호와 같다 ․...,1114
5,1,시행일,제 1 조 ( 시행일 ) 이 규정은 발령한 날부터 시행한다 . 제 2 조 ( 경과조...,3099
6,1,호,"제 1 조 ( 호 ) 를 제 1 조 ( 호 ) 의 2 로 하고 , 제 1 조 ( 호...",64
7,1,호의 경우 : 1.,제 1 조 ( 호의 경우 : 1. )\n\n- □ 조 항 호 또는 단서...,3036
8,2,폐지규정,제 2 조 ( 폐지규정 ) ○○○ 규정은 이를 폐지한다 .\n\n- 폐지규정안 형식...,1220


saved: parsed_output/13.제규정 관리규정(규정 제12호)_2016.10.17.articles.csv
saved: parsed_output/13.제규정 관리규정(규정 제12호)_2016.10.17.articles.xlsx


In [ ]:
# 8) Docling이 인식한 표를 CSV/XLSX로 저장
# PDF 변환 품질에 따라 표 인식 결과는 달라질 수 있습니다.

import pandas as pd
from pathlib import Path

tables_dir = OUT_DIR / "tables"
tables_dir.mkdir(exist_ok=True)

table_items = getattr(doc, "tables", []) or []
print("tables:", len(table_items))

table_paths = []
for idx, table in enumerate(table_items, start=1):
    try:
        df = table.export_to_dataframe()
    except Exception as e1:
        try:
            df = table.data.to_dataframe()
        except Exception as e2:
            print(f"table {idx} dataframe export failed:", repr(e1), repr(e2))
            continue

    csv_path = tables_dir / f"table_{idx:03d}.csv"
    xlsx_path = tables_dir / f"table_{idx:03d}.xlsx"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    df.to_excel(xlsx_path, index=False)
    table_paths.append((csv_path, xlsx_path))
    print("saved:", csv_path, xlsx_path)
    display(df.head())

table_paths[:5]

In [ ]:
# 9) 간단 품질 체크: 제목/조문/표 개수 확인

import re

print("Docling input:", source_for_docling)
print("Markdown chars:", len(markdown))
print("Plain text chars:", len(text))
print("Articles:", len(articles_df))
print("Tables:", len(table_items))

# 조문 제목만 빠르게 보기
for _, row in articles_df.head(30).iterrows():
    print(f"제{row['article_no']}조({row['title']})")

In [ ]:
# 10) ZIP으로 결과 묶기

import shutil
from pathlib import Path

zip_base = Path("parsed_output_bundle")
zip_path = shutil.make_archive(str(zip_base), "zip", OUT_DIR)

print("ZIP:", zip_path)

## 자주 나는 오류

### 1. PDF 변환 실패
- `libreoffice-h2orestart` 또는 `.oxt` 확장이 설치되지 않았을 수 있습니다.
- HWP가 암호화되어 있거나 오래된/특수한 HWP 버전일 수 있습니다.
- Codespaces 컨테이너를 재시작한 뒤 설치 셀부터 다시 실행해보세요.

### 2. 글자가 네모/깨짐으로 보임
- `fonts-nanum`, `fonts-noto-cjk`, `fonts-unfonts-core` 설치를 확인하세요.
- 옛한글/특수 글리프는 폰트 영향이 큽니다.

### 3. 표가 잘 안 잡힘
- HWP → PDF 변환 과정에서 표 선/셀 구조가 이미지처럼 변하면 Docling 표 추출 품질이 떨어질 수 있습니다.
- 이 경우 HWPX 변환 또는 한컴/LibreOffice에서 DOCX로 저장 후 Docling에 넣는 대안을 시도하세요.